# Pré-processamento textual — ParlaMint-PT

Preparação de texto partilhada entre experiências de deteção de drift. Cada linha continua a representar uma intervenção; os metadados e o texto original são preservados.

## Duas representações do texto

- `texto_embeddings`: texto com limpeza mínima para modelos contextuais; preserva maiúsculas, acentos, pontuação e números.
- `texto_lexical`: texto em minúsculas, tokenizado e lematizado para descritores e análises lexicais, com remoção de stopwords. Algumas palavras de negação são preservadas.

Estas são opções iniciais a validar. Uma lista provisória de termos procedimentais é removida apenas da representação lexical. Intervenções com menos de 10 palavras são excluídas antes da lematização.

In [1]:
from pathlib import Path
import re

import pandas as pd
import spacy

## 1. Carregar intervenções e metadados

A entrada é a tabela criada em `pre_processing.ipynb` após a junção do texto.

In [2]:
possible_paths = [
    Path("../../data/parlamint_pt_merged.csv"),
    Path("data/parlamint_pt_merged.csv"),
]

input_file = next((path for path in possible_paths if path.exists()), None)
if input_file is None:
    raise FileNotFoundError("Não foi encontrado data/parlamint_pt_merged.csv.")

metadata = pd.read_csv(input_file, dtype="string")
required_columns = {"ID", "Date", "texto"}
missing_columns = required_columns - set(metadata.columns)
if missing_columns:
    raise ValueError(f"Faltam colunas necessárias: {sorted(missing_columns)}")
if metadata["ID"].isna().any() or metadata["ID"].duplicated().any():
    raise ValueError("Os IDs devem estar preenchidos e ser únicos por intervenção.")

print(f"Ficheiro: {input_file}")
print(f"Intervenções: {len(metadata):,}")
print(f"Textos ausentes: {metadata['texto'].isna().sum():,}")

Ficheiro: ../../data/parlamint_pt_merged.csv
Intervenções: 179,711
Textos ausentes: 0


## 2. Limpeza mínima para embeddings

Removem-se apenas as marcas editoriais delimitadas por `[[` e `]]` e normaliza-se o espaço em branco. A coluna `texto` mantém o conteúdo original.

In [3]:
def preparar_texto_embeddings(texto):
    if pd.isna(texto):
        return ""
    texto = re.sub(r"\[\[.*?\]\]", " ", str(texto), flags=re.DOTALL)
    return re.sub(r"\s+", " ", texto).strip()

metadata["texto_embeddings"] = metadata["texto"].map(preparar_texto_embeddings)
print(f"Textos vazios após limpeza: {metadata['texto_embeddings'].eq('').sum():,}")

Textos vazios após limpeza: 0


### Filtrar intervenções curtas

Excluem-se intervenções com menos de 10 palavras no texto com limpeza mínima, antes da remoção de stopwords e da lematização. Números e palavras com hífen ou apóstrofo contam como palavras.

In [4]:
MIN_PALAVRAS = 10
metadata["n_palavras"] = metadata["texto_embeddings"].str.count(r"\b\w+(?:[-’']\w+)*\b")

intervencoes_antes = len(metadata)
metadata = metadata.loc[metadata["n_palavras"] >= MIN_PALAVRAS].copy()
intervencoes_removidas = intervencoes_antes - len(metadata)

print(f"Intervenções antes do filtro: {intervencoes_antes:,}")
print(f"Intervenções removidas: {intervencoes_removidas:,}")
if intervencoes_antes:
    print(f"Percentagem removida: {100 * intervencoes_removidas / intervencoes_antes:.1f}%")
print(f"Intervenções restantes: {len(metadata):,}")

Intervenções antes do filtro: 179,711
Intervenções removidas: 60,298
Percentagem removida: 33.6%
Intervenções restantes: 119,413


## 3. Análise de frequência antes da lematização

A análise usa as intervenções que passaram o filtro de comprimento, em minúsculas apenas para contagem. Mostra ocorrências e número de intervenções em que cada token aparece, além de sequências de duas palavras. A vista sem stopwords é apenas exploratória: não altera nem remove texto.

Frequência, por si só, não demonstra que uma palavra seja procedural. Inspeciona as intervenções em contexto antes de propor uma lista de remoção.

In [5]:
from collections import Counter
from spacy.lang.pt.stop_words import STOP_WORDS

frequencia_tokens = Counter()
documentos_por_token = Counter()
frequencia_bigramas = Counter()
documentos_por_bigrama = Counter()
padrao_token = re.compile(r"\b\w+(?:[-’']\w+)*\b")

for texto in metadata["texto_embeddings"]:
    tokens = padrao_token.findall(texto.casefold())
    bigramas = [f"{primeiro} {segundo}" for primeiro, segundo in zip(tokens, tokens[1:])]
    frequencia_tokens.update(tokens)
    documentos_por_token.update(set(tokens))
    frequencia_bigramas.update(bigramas)
    documentos_por_bigrama.update(set(bigramas))

def tabela_frequencias(frequencias, documentos, nome):
    tabela = pd.DataFrame(frequencias.items(), columns=[nome, "ocorrencias"])
    tabela["intervencoes"] = tabela[nome].map(documentos).astype(int)
    tabela["percentagem_intervencoes"] = (
        100 * tabela["intervencoes"] / len(metadata)
    ).round(2) if len(metadata) else 0.0
    return tabela.sort_values(["ocorrencias", nome], ascending=[False, True]).reset_index(drop=True)

frequencias = tabela_frequencias(frequencia_tokens, documentos_por_token, "token")
bigramas = tabela_frequencias(frequencia_bigramas, documentos_por_bigrama, "bigrama")
stopwords_pt = {palavra.casefold() for palavra in STOP_WORDS}
frequencias_sem_stopwords = frequencias.loc[~frequencias["token"].isin(stopwords_pt)]

print("30 tokens mais frequentes — incluindo stopwords:")
display(frequencias.head(30))
print("30 tokens mais frequentes — vista sem stopwords:")
display(frequencias_sem_stopwords.head(30))
print("30 expressões consecutivas de duas palavras mais frequentes:")
display(bigramas.head(30))

30 tokens mais frequentes — incluindo stopwords:


,token,ocorrencias,intervencoes,percentagem_intervencoes
0,de,965549,101145,84.70
1,que,862738,105312,88.19
2,a,741616,99684,83.48
3,e,666222,96758,81.03
4,o,600936,97871,81.96
5,do,348305,80124,67.10
6,é,314362,83912,70.27
7,da,285466,69995,58.62
8,para,267830,75181,62.96
9,não,267072,78689,65.90


30 tokens mais frequentes — vista sem stopwords:


,token,ocorrencias,intervencoes,percentagem_intervencoes
12,sr,203599,81526,68.27
26,governo,95038,42238,35.37
29,presidente,71165,59718,50.01
32,srs,62908,40406,33.84
35,deputados,59360,41603,34.84
36,ª,58971,25749,21.56
41,há,53426,30884,25.86
51,país,39865,22849,19.13
55,deputado,38101,21250,17.80
57,hoje,36633,24359,20.40


30 expressões consecutivas de duas palavras mais frequentes:


,bigrama,ocorrencias,intervencoes,percentagem_intervencoes
0,que o,74948,43501,36.43
1,sr presidente,57133,50377,42.19
2,srs deputados,54623,39748,33.29
3,o que,52966,32603,27.30
4,que é,45063,29115,24.38
5,é que,42568,27096,22.69
6,que a,40247,27180,22.76
7,que não,40120,27377,22.93
8,o governo,39648,22899,19.18
9,sr ª,37673,22500,18.84


## 4. Lematização e stopwords em português

Requer spaCy e o modelo `pt_core_news_sm`. O modelo é uma escolha inicial; a qualidade das lematizações deve ser inspecionada em português europeu. Os números são preservados e as negações definidas abaixo não são removidas como stopwords. A lista procedural é provisória e aplicada apenas a `texto_lexical`; `texto_embeddings` mantém o texto contextual.

In [6]:
SPACY_MODEL = "pt_core_news_sm"
NEGACOES_A_PRESERVAR = {"não", "nem", "nunca", "jamais", "sem"}
TERMOS_PROCEDIMENTAIS = {
    "sr", "sra", "srs", "sr.ª", "ª", "senhor", "senhora", "senhores", "senhoras",
    "deputado", "deputados", "deputada", "deputadas", "presidente", "vice-presidente",
    "secretário", "secretária", "excelência", "excelências", "exmo", "exma",
    "obrigado", "obrigada", "agradeço", "cumprimento", "cumprimentos",
    "aplausos", "protestos", "risos", "interrupção", "interrupções",
}
BATCH_SIZE = 128

try:
    nlp = spacy.load(SPACY_MODEL, disable=["parser", "ner"])
except OSError as error:
    raise OSError(
        f"O modelo spaCy {SPACY_MODEL!r} não está instalado no kernel atual. "
        "Instala-o antes de continuar."
    ) from error

from spacy.matcher import PhraseMatcher
matcher_procedimental = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher_procedimental.add("PROCEDIMENTAL", [nlp.make_doc(termo) for termo in sorted(TERMOS_PROCEDIMENTAIS)])

print(f"Modelo linguístico: {SPACY_MODEL}")

Modelo linguístico: pt_core_news_sm


In [7]:
def lemas_relevantes(doc):
    lemas = []
    indices_procedimentais = {i for _, inicio, fim in matcher_procedimental(doc) for i in range(inicio, fim)}
    for token in doc:
        if token.i in indices_procedimentais:
            continue
        if token.is_space or token.is_punct or not (token.is_alpha or token.like_num):
            continue
        forma = token.text.lower()
        lema = token.lemma_.strip().lower() or forma
        if forma in TERMOS_PROCEDIMENTAIS or lema in TERMOS_PROCEDIMENTAIS:
            continue
        if token.is_stop and forma not in NEGACOES_A_PRESERVAR and lema not in NEGACOES_A_PRESERVAR:
            continue
        lemas.append(lema)
    return lemas

tokens_lexicais = [
    lemas_relevantes(doc)
    for doc in nlp.pipe(metadata["texto_embeddings"].tolist(), batch_size=BATCH_SIZE)
]
metadata["texto_lexical"] = [" ".join(tokens) for tokens in tokens_lexicais]
metadata["n_tokens_lexicais"] = [len(tokens) for tokens in tokens_lexicais]

print(f"Intervenções sem tokens lexicais: {metadata['n_tokens_lexicais'].eq(0).sum():,}")

Intervenções sem tokens lexicais: 16


## 5. Inspeção e validação

Verifica se os acentos foram preservados, se a lematização é adequada e se a limpeza não eliminou conteúdo relevante.

In [8]:
display(
    metadata[["ID", "texto", "texto_embeddings", "texto_lexical", "n_tokens_lexicais"]]
    .sample(n=min(10, len(metadata)), random_state=42)
)

display(metadata["n_tokens_lexicais"].describe())

,ID,texto,texto_embeddings,texto_lexical,n_tokens_lexicais
153296,ParlaMint-PT_2023-05-10.u38,"Na verdade, a ideia da direita — e a ideia do ...","Na verdade, a ideia da direita — e a ideia do ...",verdade ideia ideia ps preciso exatamente gast...,41
90693,ParlaMint-PT_2020-02-03.u38,O Partido Socialista apresentou um conjunto de...,O Partido Socialista apresentou um conjunto de...,partido socialista apresentar conjunto propost...,50
103173,ParlaMint-PT_2020-10-22.u2,"Sr. Presidente, Sr.as e Srs. Deputados, deram ...","Sr. Presidente, Sr.as e Srs. Deputados, deram ...",dar entrada mesa admitir seguinte iniciativa l...,46
117666,ParlaMint-PT_2021-11-26.u87,"Sr. Presidente, para anunciar que sobre este p...","Sr. Presidente, para anunciar que sobre este p...",anunciar projeto voto anterior entregaremo dec...,7
76829,ParlaMint-PT_2019-03-15.u352,"Acresce que, com as vossas legislações, andara...","Acresce que, com as vossas legislações, andara...",acrescer legislação andar brincar legionella m...,36
60996,ParlaMint-PT_2018-05-18.u290,"Sr. Presidente, Sr.as e Srs. Deputados, o voto...","Sr. Presidente, Sr.as e Srs. Deputados, o voto...",voto seguinte teor passado dia 14 maio dia mor...,153
150966,ParlaMint-PT_2023-04-20.u119,"É por isso que falar do juiz A ou B, no proces...","É por isso que falar do juiz A ou B, no proces...",falar juiz b processo c d ser culpa governo fr...,112
124222,ParlaMint-PT_2022-06-29.u214,"Aquilo que acabou de dizer, na sequência dos p...","Aquilo que acabou de dizer, na sequência dos p...",acabar sequência projeto lei estar apresentar ...,127
114623,ParlaMint-PT_2021-10-01.u175,"Sr. Presidente, para assinalar que entregaremo...","Sr. Presidente, para assinalar que entregaremo...",assinalar entregaremos declaração voto escrito...,6
178567,ParlaMint-PT_2024-01-09.u377,"Sr.ª Presidente, cumprimento-a a si, bem como ...","Sr.ª Presidente, cumprimento-a a si, bem como ...",si colocar perspetiva espaço união europeia de...,252


count    119413.000000
mean         76.121528
std          94.590229
min           0.000000
25%          15.000000
50%          38.000000
75%         103.000000
max        1384.000000
Name: n_tokens_lexicais, dtype: float64

## 6. Guardar os dois corpus

Os dois CSV têm os mesmos IDs e metadados, com uma coluna `texto_modelo` diferente: texto contextual com limpeza mínima ou texto lexical lematizado, sem stopwords e termos procedimentais. Intervenções cujo texto lexical ficou vazio são retiradas de ambos, para preservar a comparabilidade. Os CSV existentes não são substituídos.


In [9]:
output_dir = input_file.resolve().parent
contextual_file = output_dir / "parlamint_pt_corpus_contextual.csv"
lexical_file = output_dir / "parlamint_pt_corpus_lexical.csv"
existentes = [str(p) for p in (contextual_file, lexical_file) if p.exists()]
if existentes:
    raise FileExistsError(f"Ficheiros já existentes, não substituídos: {existentes}")

validos = metadata["texto_lexical"].fillna("").str.strip().ne("")
print(f"Intervenções com texto lexical vazio removidas de ambos: {(~validos).sum():,}")
corpus = metadata.loc[validos].sort_values("ID").reset_index(drop=True)
colunas_comuns = corpus.drop(columns=["texto_embeddings", "texto_lexical"])
contextual = colunas_comuns.copy()
lexical = colunas_comuns.copy()
contextual["texto_modelo"] = corpus["texto_embeddings"]
lexical["texto_modelo"] = corpus["texto_lexical"]
assert contextual["ID"].equals(lexical["ID"])
contextual.to_csv(contextual_file, index=False, encoding="utf-8")
lexical.to_csv(lexical_file, index=False, encoding="utf-8")
print(f"Corpus contextual: {contextual_file}")
print(f"Corpus lexical: {lexical_file}")
print(f"Intervenções em cada corpus: {len(corpus):,}")


Intervenções com texto lexical vazio removidas de ambos: 16
Corpus contextual: /Users/alexandresousa/Desktop/Tese/data/parlamint_pt_corpus_contextual.csv
Corpus lexical: /Users/alexandresousa/Desktop/Tese/data/parlamint_pt_corpus_lexical.csv
Intervenções em cada corpus: 119,397


## 7. Embeddings das duas versões com MPS

Esta secção pode ser executada num kernel novo depois de guardar os CSV, sem voltar a lematizar. Por defeito usa o piloto de 2015–2017; altera `ANOS_PILOTO` para `None` para usar todo o corpus. O mesmo BERTimbau base e parâmetros são usados para ambos. Os textos com mais de 512 tokens do modelo são truncados. Cada versão fica em pasta própria, com blocos retomáveis e uma assinatura que verifica documentos, textos e parâmetros. Não apaga nem substitui os embeddings do piloto anterior.


In [10]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, models

ANOS_PILOTO = [2015, 2016, 2017]  # None para todo o corpus
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_SEQ_LENGTH = 512
EMBEDDING_BATCH_SIZE = 4
CHUNK_SIZE = 512

data_paths = [Path("../../data"), Path("data")]
data_dir_embeddings = next((p.resolve() for p in data_paths if (p / "parlamint_pt_corpus_contextual.csv").exists()), None)
if data_dir_embeddings is None:
    raise FileNotFoundError("Guarda primeiro os dois corpus na secção 6.")
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS não está disponível neste kernel. Usa um kernel PyTorch com suporte a MPS no Mac.")

corpora = {}
for versao in ("contextual", "lexical"):
    ficheiro = data_dir_embeddings / f"parlamint_pt_corpus_{versao}.csv"
    dados = pd.read_csv(ficheiro, dtype="string", keep_default_na=False)
    if not {"ID", "Date", "texto_modelo"}.issubset(dados.columns):
        raise ValueError(f"Colunas necessárias em falta: {ficheiro}")
    if ANOS_PILOTO is not None:
        anos = pd.to_datetime(dados["Date"], errors="coerce").dt.year
        dados = dados.loc[anos.isin(ANOS_PILOTO)].copy()
    dados = dados.sort_values("ID").reset_index(drop=True)
    if dados.empty or dados["ID"].eq("").any() or dados["ID"].duplicated().any() or dados["texto_modelo"].str.strip().eq("").any():
        raise ValueError(f"Corpus vazio ou com IDs/textos inválidos: {ficheiro}")
    corpora[versao] = dados
if not corpora["contextual"]["ID"].equals(corpora["lexical"]["ID"]):
    raise ValueError("Os dois corpus não têm os mesmos IDs na mesma ordem.")
print(f"Documentos por versão: {len(corpora['contextual']):,}; anos: {ANOS_PILOTO or 'todos'}")

transformer = models.Transformer(MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH)
pooling = models.Pooling(transformer.get_word_embedding_dimension(), pooling_mode_mean_tokens=True,
                         pooling_mode_cls_token=False, pooling_mode_max_tokens=False)
embedding_model = SentenceTransformer(modules=[transformer, pooling], device="mps")
print(f"Modelo: {MODEL_NAME}; dispositivo: mps")


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/2t/mv0q1c7j2z97cgvtlhbfrhhm0000gn/T/ipykernel_12666/2655445968.py:7: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models


Documentos por versão: 36,258; anos: [2015, 2016, 2017]


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39800.99it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/var/folders/2t/mv0q1c7j2z97cgvtlhbfrhhm0000gn/T/ipykernel_12666/2655445968.py:40: FutureWarning: The `get_w

Modelo: neuralmind/bert-base-portuguese-cased; dispositivo: mps


In [11]:
def assinatura_corpus(ids, textos, versao):
    h = hashlib.sha256()
    config = {"modelo": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "pooling": "mean",
              "normalize_embeddings": True, "versao": versao, "chunk_size": CHUNK_SIZE}
    h.update(json.dumps(config, sort_keys=True).encode())
    for identificador, texto in zip(ids, textos):
        for valor in (identificador, texto):
            dados = valor.encode("utf-8")
            h.update(len(dados).to_bytes(8, "big"))
            h.update(dados)
    return h.hexdigest()

nome_execucao = "todos_anos" if ANOS_PILOTO is None else "anos_" + "_".join(map(str, ANOS_PILOTO))
base = data_dir_embeddings / "embeddings_bertimbau" / nome_execucao
for versao, dados in corpora.items():
    ids = dados["ID"].astype(str).tolist()
    textos = dados["texto_modelo"].astype(str).tolist()
    assinatura = assinatura_corpus(ids, textos, versao)
    pasta = base / versao
    pasta.mkdir(parents=True, exist_ok=True)
    manifesto = pasta / "manifesto.json"
    if manifesto.exists():
        anterior = json.loads(manifesto.read_text(encoding="utf-8"))
        if anterior["assinatura"] != assinatura:
            raise ValueError(f"Corpus ou parâmetros diferentes em {pasta}; escolhe uma pasta nova.")
    else:
        if any(pasta.iterdir()):
            raise ValueError(f"Pasta não vazia sem manifesto: {pasta}")
        manifesto.write_text(json.dumps({"assinatura": assinatura, "modelo": MODEL_NAME,
            "versao": versao, "documentos": len(ids), "chunk_size": CHUNK_SIZE},
            ensure_ascii=False, indent=2), encoding="utf-8")
    for inicio in range(0, len(ids), CHUNK_SIZE):
        fim = min(inicio + CHUNK_SIZE, len(ids))
        destino = pasta / f"bloco_{inicio:08d}_{fim:08d}.npz"
        ids_bloco = np.asarray(ids[inicio:fim], dtype=str)
        if destino.exists():
            with np.load(destino, allow_pickle=False) as bloco:
                if not np.array_equal(bloco["document_ids"], ids_bloco) or bloco["embeddings"].shape[0] != len(ids_bloco):
                    raise ValueError(f"Bloco incompatível: {destino}")
            continue
        embeddings = embedding_model.encode(textos[inicio:fim], batch_size=EMBEDDING_BATCH_SIZE,
                                             show_progress_bar=True, normalize_embeddings=True)
        temporario = destino.with_suffix(".tmp.npz")
        np.savez_compressed(temporario, document_ids=ids_bloco, embeddings=embeddings)
        temporario.replace(destino)
        print(f"{versao}: {fim:,}/{len(ids):,} documentos guardados")
    print(f"Embeddings concluídos: {pasta}")


Batches: 100%|██████████| 128/128 [00:17<00:00,  7.18it/s]


contextual: 512/36,258 documentos guardados


Batches: 100%|██████████| 128/128 [00:19<00:00,  6.64it/s]


contextual: 1,024/36,258 documentos guardados


Batches: 100%|██████████| 128/128 [00:18<00:00,  6.93it/s]


contextual: 1,536/36,258 documentos guardados


... (274 outputs left out)

Batches: 100%|██████████| 128/128 [00:19<00:00,  6.56it/s]


lexical: 35,840/36,258 documentos guardados


Batches: 100%|██████████| 105/105 [00:27<00:00,  3.88it/s]

lexical: 36,258/36,258 documentos guardados
Embeddings concluídos: /Users/alexandresousa/Desktop/Tese/data/embeddings_bertimbau/anos_2015_2016_2017/lexical


Os blocos contêm os IDs e os embeddings correspondentes. O notebook BERTilda existente ainda não os carrega automaticamente; será necessário adaptá-lo antes de comparar os tópicos. A lematização e a remoção de stopwords e termos procedimentais variam em conjunto nesta comparação.
